# Task 1.1 published figure and TPGT training

Use a **GPU** runtime. Run the install cell first (it clones the repo and mounts Drive), accept the Drive prompt, then run the training cell.

The training cell draws the published Task 1.1 panel and trains **TPGT** (the type-pair graph Transformer with a separate weight set at each correction layer) on the IEEE 118-bus data used by GENCO Base, PFNet, CANOS-PF, and GNS-S. Data path: `My Drive/Foundation Model/vendor/pfdelta_task1.1`. Checkpoints are saved to `Foundation Model/tpgt_task11` (`seed_0`, `seed_1`, `seed_2`) and TPGT tables to `Foundation Model/results`. A finished seed is skipped if the training cell is run again.

The TPGT sizes are implementation choices for Colab, not the authors' hyperparameters: hidden size 32, 2 heads, 3 unshared layers, batch 16, Adam learning rate 1e-4, 5 epochs, seeds 0, 1, and 2. IEEE 118 pair features use a slack-rooted spanning tree because the draft's path impedance assumes a radial feeder.

In [ ]:
import importlib
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/genco-figure4-task11")
REPO_URL = "https://github.com/alitasavori/genco-figure4-task11.git"


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not (REPO / "train_tpgt_task11.py").is_file():
        subprocess.check_call(["git", "clone", REPO_URL, str(REPO)])
    from google.colab import drive

    drive.mount("/content/drive")

missing = []
for module, package in (
    ("numpy", "numpy"),
    ("torch", "torch"),
    ("matplotlib", "matplotlib"),
    ("pyarrow", "pyarrow"),
    ("huggingface_hub", "huggingface_hub"),
):
    try:
        importlib.import_module(module)
    except ImportError:
        missing.append(package)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("Required packages are already installed.")


In [ ]:
import os
import subprocess
from pathlib import Path

REPO = Path("/content/genco-figure4-task11")
REPO_URL = "https://github.com/alitasavori/genco-figure4-task11.git"
COLAB_DATA = "/content/drive/MyDrive/Foundation Model/vendor/pfdelta_task1.1"
WINDOWS_DATA = Path(r"K:\My Drive\Foundation Model\vendor\pfdelta_task1.1")


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


here = Path.cwd()
scripts_here = (here / "plot_task1_1.py").is_file() and (here / "train_tpgt_task11.py").is_file()
if not scripts_here:
    if on_colab():
        if not REPO.is_dir():
            subprocess.check_call(["git", "clone", REPO_URL, str(REPO)])
        os.chdir(REPO)
    elif (here / "genco" / "plot_task1_1.py").is_file():
        os.chdir(here / "genco")
    else:
        raise FileNotFoundError(
            "plot_task1_1.py and train_tpgt_task11.py are not in this folder."
        )

if on_colab():
    if not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive

        drive.mount("/content/drive")
    os.environ["GENCO_PFDELTA_TASK11"] = COLAB_DATA
elif WINDOWS_DATA.is_dir():
    os.environ["GENCO_PFDELTA_TASK11"] = str(WINDOWS_DATA)

%run plot_task1_1.py --published
%run train_tpgt_task11.py
